Renamed cell titles to 2-word descriptive names in the table of contents
*Co-authored with CoCo*

# Generate and Load Sample Data

This notebook generates synthetic payment data for the finserv fraud detection demo and loads it into Snowflake.

## What this notebook does:
- **Creates 2,000 customers** with realistic spending profiles and account tiers
- **Creates 500 merchants** across categories like e-commerce, travel, and retail
- **Generates 10,000 transactions** with realistic payment patterns and fraud indicators
- **Loads all data directly** into Snowflake tables

## Prerequisites:
1. Run `01_env_setup.sql` first to create the environment and tables
2. Ensure you're connected to the `FINSERV_FRAUD_DEMO.RAW_DATA` schema
3. Running in **Snowflake Notebooks** environment

## Data Features:
- **Fraud Detection Ready**: Transactions include fraud patterns for ML training (~4% fraud rate)
- **Realistic Distributions**: Data follows real-world payment fraud patterns
- **Time-based**: 50% of data in past 30 days for recency analysis

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    print("Running in Snowflake Notebooks")
except:
    from snowflake_connection import get_local_session
    session = get_local_session()
    print("Running locally")

print(f"Account: {session.get_current_account()}")
print(f"User: {session.get_current_user()}")
print(f"Warehouse: {session.get_current_warehouse()}")

session.use_database("FINSERV_FRAUD_DEMO")
session.use_schema("RAW_DATA")

np.random.seed(42)
random.seed(42)

print(f"Ready to work with {session.get_current_database()}.{session.get_current_schema()}")
print("="*60)

In [ ]:
# Configuration
NUM_CUSTOMERS = 2000
NUM_MERCHANTS = 500
NUM_TRANSACTIONS = 10000

# Date range - past 90 days with 50% in past 30 days
end_date = datetime.now()
start_date_90d = end_date - timedelta(days=90)
start_date_30d = end_date - timedelta(days=30)

print(f"Generating sample data from {start_date_90d.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"Customers: {NUM_CUSTOMERS:,}")
print(f"Merchants: {NUM_MERCHANTS:,}")
print(f"Transactions: {NUM_TRANSACTIONS:,}")

In [ ]:
# Generate customer profiles
print("Generating customer profiles...")

customer_profiles = []
for customer_id in range(1, NUM_CUSTOMERS + 1):
    account_tier = np.random.choice(['standard', 'premium', 'private'], p=[0.70, 0.20, 0.10])

    if account_tier == 'standard':
        avg_monthly_spend = np.random.uniform(500, 2000)
        num_linked_cards = np.random.randint(1, 3)
        account_age_days = np.random.randint(30, 1825)
    elif account_tier == 'premium':
        avg_monthly_spend = np.random.uniform(2000, 8000)
        num_linked_cards = np.random.randint(2, 5)
        account_age_days = np.random.randint(180, 3650)
    else:  # private
        avg_monthly_spend = np.random.uniform(8000, 50000)
        num_linked_cards = np.random.randint(3, 8)
        account_age_days = np.random.randint(365, 3650)

    customer_profiles.append({
        'customer_id': customer_id,
        'account_tier': account_tier,
        'account_age_days': account_age_days,
        'avg_monthly_spend': round(avg_monthly_spend, 2),
        'num_linked_cards': num_linked_cards
    })

print(f"Generated {len(customer_profiles)} customer profiles")

In [ ]:
# Generate merchant data
print("Generating merchant data...")

merchant_categories = ['e_commerce', 'food_delivery', 'travel', 'retail', 'gas_station', 'entertainment', 'healthcare']
category_risk = {
    'e_commerce': 'high', 'travel': 'high', 'food_delivery': 'low',
    'retail': 'low', 'gas_station': 'medium', 'entertainment': 'medium', 'healthcare': 'low'
}
country_codes = ['US'] * 80 + ['CA', 'GB', 'AU', 'DE', 'FR', 'MX', 'BR', 'SG', 'JP', 'IN'] * 2
merchant_name_prefixes = {
    'e_commerce': ['ShopNow', 'QuickBuy', 'NetCart', 'SwiftShop', 'EasyBuy'],
    'food_delivery': ['QuickBite', 'FoodRush', 'DashEats', 'FastFork', 'MealDrop'],
    'travel': ['JetSet', 'AirNow', 'SwiftFly', 'TravelGo', 'BookTrip'],
    'retail': ['MegaMart', 'ValueStore', 'ShopSmart', 'TownMart', 'BestBuy'],
    'gas_station': ['FuelStop', 'QuickGas', 'RoadFuel', 'EasyFill', 'SpeedFuel'],
    'entertainment': ['StreamNow', 'PlayZone', 'FunPass', 'TicketHub', 'GamePlus'],
    'healthcare': ['MedPlus', 'QuickCare', 'PharmaDirect', 'HealthHub', 'RxExpress']
}

merchants_data = []
for merchant_id in range(1, NUM_MERCHANTS + 1):
    category = np.random.choice(merchant_categories)
    country = np.random.choice(country_codes)
    prefix = np.random.choice(merchant_name_prefixes[category])
    merchants_data.append({
        'merchant_id': merchant_id,
        'merchant_name': f"{prefix} {merchant_id}",
        'merchant_category': category,
        'country_code': country,
        'risk_tier': category_risk[category]
    })

merchants_df_map = {m['merchant_id']: m for m in merchants_data}
print(f"Generated {len(merchants_data)} merchants")

In [ ]:
# Generate transaction data
print("Generating transaction data...")

device_types = ['mobile_app', 'web_browser', 'pos_terminal', 'atm']
channels = {'mobile_app': 'online', 'web_browser': 'online', 'pos_terminal': 'in_person', 'atm': 'atm'}
customer_profiles_map = {c['customer_id']: c for c in customer_profiles}

transactions_data = []
transaction_id = 1

for _ in range(NUM_TRANSACTIONS):
    customer_id = np.random.randint(1, NUM_CUSTOMERS + 1)
    merchant_id = np.random.randint(1, NUM_MERCHANTS + 1)
    profile = customer_profiles_map[customer_id]
    merchant = merchants_df_map[merchant_id]

    # Transaction datetime - 50% in past 30 days
    if np.random.random() < 0.5:
        days_ago = np.random.randint(0, 30)
    else:
        days_ago = np.random.randint(30, 90)

    txn_datetime = end_date - timedelta(
        days=days_ago,
        hours=np.random.randint(0, 24),
        minutes=np.random.randint(0, 60)
    )

    # Fraud probability: base 4%, higher for high-risk merchants
    fraud_probability = 0.08 if merchant['risk_tier'] == 'high' else 0.04
    is_fraud = np.random.random() < fraud_probability

    avg_spend = profile['avg_monthly_spend'] / 30  # daily average

    if is_fraud:
        # Fraud pattern: amount is 3-8x the customer's normal daily average
        amount = round(avg_spend * np.random.uniform(3.0, 8.0), 2)
        # Fraud more likely on international merchants
        country_code = np.random.choice(
            ['GB', 'AU', 'DE', 'FR', 'SG', 'JP', 'BR'] if np.random.random() < 0.6
            else [merchant['country_code']]
        )
        device_type = np.random.choice(['mobile_app', 'web_browser'], p=[0.5, 0.5])
    else:
        amount = round(max(avg_spend * np.random.uniform(0.1, 2.0), 1.0), 2)
        country_code = merchant['country_code']
        device_type = np.random.choice(device_types, p=[0.45, 0.30, 0.20, 0.05])

    transactions_data.append({
        'transaction_id': transaction_id,
        'customer_id': customer_id,
        'merchant_id': merchant_id,
        'amount': amount,
        'merchant_category': merchant['merchant_category'],
        'country_code': country_code,
        'device_type': device_type,
        'channel': channels[device_type],
        'transaction_datetime': txn_datetime,
        'is_fraud': is_fraud
    })
    transaction_id += 1

fraud_count = sum(1 for t in transactions_data if t['is_fraud'])
print(f"Generated {len(transactions_data)} transactions")
print(f"Fraud rate: {fraud_count/len(transactions_data)*100:.1f}% ({fraud_count} fraudulent transactions)")

## Data Loading

Load the generated data into Snowflake tables.

In [ ]:
tables_loaded = []

# Load customers
print("Loading customers data...")
try:
    session.sql("TRUNCATE TABLE CUSTOMERS").collect()
    insert_values = []
    for row in customer_profiles:
        values = f"({row['customer_id']}, '{row['account_tier']}', {row['account_age_days']}, {row['avg_monthly_spend']}, {row['num_linked_cards']})"
        insert_values.append(values)
    batch_size = 1000
    for i in range(0, len(insert_values), batch_size):
        batch = insert_values[i:i+batch_size]
        session.sql(f"INSERT INTO CUSTOMERS VALUES {','.join(batch)}").collect()
    print(f"Customers loaded successfully! ({len(customer_profiles)} rows)")
    tables_loaded.append("CUSTOMERS")
except Exception as e:
    print(f"Error loading customers: {e}")

In [ ]:
# Load merchants
print("Loading merchants data...")
try:
    session.sql("TRUNCATE TABLE MERCHANTS").collect()
    insert_values = []
    for row in merchants_data:
        values = f"({row['merchant_id']}, '{row['merchant_name']}', '{row['merchant_category']}', '{row['country_code']}', '{row['risk_tier']}')"
        insert_values.append(values)
    batch_size = 1000
    for i in range(0, len(insert_values), batch_size):
        batch = insert_values[i:i+batch_size]
        session.sql(f"INSERT INTO MERCHANTS VALUES {','.join(batch)}").collect()
    print(f"Merchants loaded successfully! ({len(merchants_data)} rows)")
    tables_loaded.append("MERCHANTS")
except Exception as e:
    print(f"Error loading merchants: {e}")

In [ ]:
# Load transactions
print("Loading transaction data...")
try:
    session.sql("TRUNCATE TABLE TRANSACTIONS").collect()
    insert_values = []
    for row in transactions_data:
        values = f"({row['transaction_id']}, {row['customer_id']}, {row['merchant_id']}, {row['amount']}, '{row['merchant_category']}', '{row['country_code']}', '{row['device_type']}', '{row['channel']}', '{row['transaction_datetime']}', {str(row['is_fraud']).upper()})"
        insert_values.append(values)
    batch_size = 1000
    for i in range(0, len(insert_values), batch_size):
        batch = insert_values[i:i+batch_size]
        session.sql(f"INSERT INTO TRANSACTIONS VALUES {','.join(batch)}").collect()
    print(f"Transactions loaded successfully! ({len(transactions_data)} rows)")
    tables_loaded.append("TRANSACTIONS")
except Exception as e:
    print(f"Error loading transactions: {e}")

In [ ]:
# Verify data loaded
if tables_loaded:
    print(f"Data verification for {len(tables_loaded)} tables:")
    union_queries = [f"SELECT '{t}' as table_name, COUNT(*) as row_count FROM {t}" for t in tables_loaded]
    row_counts = session.sql(" UNION ALL ".join(union_queries)).collect()
    for row in row_counts:
        print(f"  - {row['TABLE_NAME']}: {row['ROW_COUNT']} rows")

    print("\nSample data preview:")
    for table in tables_loaded:
        print(f"\n{table} sample:")
        session.table(table).limit(3).show()

    print(f"\nSuccessfully loaded {len(tables_loaded)} tables!")
    print("You can now run the ML demo notebook.")
else:
    print("No tables were loaded. Check the errors above.")

In [ ]:
print("Data Generation Summary:")
print(f"Customers: {NUM_CUSTOMERS:,}")
print(f"Merchants: {NUM_MERCHANTS:,}")
print(f"Transactions: {len(transactions_data):,}")
fraud_count = sum(1 for t in transactions_data if t['is_fraud'])
print(f"Fraud rate: {fraud_count/len(transactions_data)*100:.1f}% ({fraud_count} fraudulent)")
print(f"Date range: {start_date_90d.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"Tables loaded: {len(tables_loaded)}")
print("\nNext step: run 03_finserv_fraud_demo.ipynb")